In [2]:
import sys
import os
REPO_URL = "https://github.com/PabloVM/JLM-jev_like_model"
REPO_NAME = "JLM-jev_like_model"
!git clone {REPO_URL}


os.chdir("/kaggle/working")


sys.path.append(f"/kaggle/working/{REPO_NAME}")


Cloning into 'JLM-jev_like_model'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 75 (delta 30), reused 59 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 258.27 KiB | 2.44 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [3]:
%cd /kaggle/working/JLM-jev_like_model
!git pull

/kaggle/working/JLM-jev_like_model
Already up to date.


In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token

In [5]:
import torch
import gc
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from pathlib import Path

from src.data import JLMTokenizer, JLMDataset, jlm_collate_fn
from src.model import JLM

In [6]:



MODEL_NAME = "Qwen/Qwen3-0.6B"
BATCH_SIZE = 2
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
SAVE_DIR = Path("./checkpoints")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [7]:
print("Loading Dataset : 'tasksource/tasksource-jev-typed-decisions'...")
train_hf = load_dataset("tasksource/tasksource-jev-typed-decisions", split="train")
test_hf = load_dataset("tasksource/tasksource-jev-typed-decisions", split="test")

def filter_long_examples(example):
    options = example.get("options") or []
    return len(options) <= 6 and len(example.get("question", "")) < 1000
    
train_hf = train_hf.filter(filter_long_examples)
test_hf = test_hf.filter(filter_long_examples)

train_hf = train_hf.shuffle(seed=42).select(range(100))
val_hf = test_hf.shuffle(seed=42).select(range(20))

print(f"Train entries : {len(train_hf)} | Validation entries : {len(test_hf)}")

Loading Dataset : 'tasksource/tasksource-jev-typed-decisions'...


README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

data/train-00000-of-00006.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

...

Generating validation split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/999832 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15000 [00:00<?, ? examples/s]

Train entries : 100 | Validation entries : 12385


In [8]:
tokenizer = JLMTokenizer(MODEL_NAME)

train_dataset = JLMDataset(train_hf,tokenizer)
val_dataset = JLMDataset(test_hf,tokenizer)

train_loader = DataLoader(
    train_dataset,batch_size=BATCH_SIZE, shuffle=True, collate_fn=jlm_collate_fn
)

val_loader = DataLoader(
    val_dataset,batch_size=BATCH_SIZE, shuffle=True, collate_fn=jlm_collate_fn
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = JLM(MODEL_NAME)

model.to(device) 

first_device = device
head_device = device

print(f"Base Model : {first_device}")
print(f"Head : {head_device}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base Model : cuda:0
Head : cuda:0


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.head.parameters(), lr=LEARNING_RATE)

In [11]:
from tqdm.auto import tqdm

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.head.train()
    total_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(dataloader, desc="Training" ,leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        position_ids = batch["position_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        choice_idx = batch["choices_end_indxs"].to(device)
        choice_mask = batch["choice_mask"].to(device)
        answer_idx = batch["answer_end_indxs"].to(device)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
            # logits (batch_size, max_choices)
            logits = model(
                input_ids=input_ids,
                position_ids=position_ids,
                attention_mask=attention_mask,
                choice_idx=choice_idx,
                choice_mask=choice_mask,
                answer_idx=answer_idx
            )

       
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

       
        pred = torch.argmax(logits, dim=-1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / len(dataloader), (correct / total) * 100

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(dataloader, desc="Eval", leave=False)
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(first_device)
        position_ids = batch["position_ids"].to(first_device)
        attention_mask = batch["attention_mask"].to(first_device)
        labels = batch["labels"].to(head_device)

        choice_idx = batch["choices_end_indxs"].to(device)
        choice_mask = batch["choice_mask"].to(device)
        answer_idx = batch["answer_end_indxs"].to(device)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
            logits = model(
                input_ids=input_ids,
                position_ids=position_ids,
                attention_mask=attention_mask,
                choice_idx=choice_idx,
                choice_mask=choice_mask,
                answer_idx=answer_idx
            )
        loss = criterion(logits, labels)

        batch_loss = 0.0
        pred = torch.argmax(logits, dim=-1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), (correct / total) * 100

In [12]:


best_val_acc = 0.0

print("\n=================== START TRAINING ===================")
for epoch in range(0, NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion,device)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")

   
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = SAVE_DIR / "best_head.pt"
        torch.save(model.head.state_dict(), save_path)
        print(f"New best validation accuracy. Weights saved: {save_path}")

print("=================== END TRAINING ===================")
print(f"Best validation accuracy: {best_val_acc:.2f}%")


=================== START TRAINING ===================


Training:   0%|          | 0/50 [00:00<?, ?it/s]

Eval:   0%|          | 0/6193 [00:00<?, ?it/s]

KeyboardInterrupt: 